# Домашнее задание 2. Ansible-роль для PostgreSQL

В работе используется Ubuntu Server 24.04 ARM64. Текущая виртуальная машина выступает Ansible-контроллером и целью `vm1`, как в примере занятия.

## 1. Окружение и зависимости

In [1]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
uname -m
. /etc/os-release
echo "$PRETTY_NAME"
ansible --version | head -n 2
ansible-galaxy collection install -r requirements.yml
ansible-galaxy collection list community.postgresql

aarch64


Ubuntu 24.04.5 LTS


ansible [core 2.21.5]
  config file = /mnt/hgfs/hw/HW2/ansible.cfg


Starting galaxy collection install process
Nothing to do. All requested collections are already inst

alled. If you want to reinstall them, consider using `--force`.



# /home/danil/.ansible/collections/ansible_collections
Collection           Version
---------------

----- -------
community.postgresql 5.0.0  

# /usr/lib/python3/dist-packages/ansible_collections
Col

lection           Version
-------------------- -------
community.postgresql 4.2.0  


Проект оформлен по структуре из лекции: inventory хранится отдельно, роль имеет `defaults`, внутренние `vars`, задачи, handlers и шаблоны. Пароли лежат в зашифрованном `vault.yml`, а локальный файл `.vault-pass` исключён из Git.

In [2]:
%%bash
cd /mnt/hgfs/hw/HW2
find . -maxdepth 5 -type f \
  ! -path './.venv/*' ! -name '.vault-pass' ! -name 'HW2.ipynb' \
  | sort

./.gitignore
./README.md
./ansible.cfg
./inventories/main/group_vars/all/vault.yml
./inventories/mai

n/host_vars/vm1.yml
./inventories/main/hosts.yaml
./postgres.play.yml
./requirements.txt
./requireme

nts.yml
./roles/postgres/README.md
./roles/postgres/defaults/main.yml
./roles/postgres/handlers/main

.yml
./roles/postgres/tasks/configure.yml
./roles/postgres/tasks/databases.yml
./roles/postgres/task

s/exporter.yml
./roles/postgres/tasks/install.yml
./roles/postgres/tasks/main.yml
./roles/postgres/t

emplates/pg_hba.conf.j2
./roles/postgres/templates/postgres-exporter.env.j2
./roles/postgres/templat

es/postgres-exporter.service.j2
./roles/postgres/templates/postgresql-hw2.service.j2
./roles/postgre

s/vars/main.yml


## 2. Проверка inventory и плейбука

In [3]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible-inventory --graph
ansible vm1 -m ping
ansible-playbook --syntax-check postgres.play.yml

@all:
  |--@ungrouped:
  |  |--vm1


vm1 | SUCCESS => {
    "ansible_facts": {
        "discovered_interpreter_python": "/usr/bin/python3

.12"
    },
    "changed": false,
    "ping": "pong"
}



playbook: postgres.play.yml


## 3. Первый запуск

Плейбук устанавливает PostgreSQL из стандартного репозитория Ubuntu, инициализирует отдельный кластер в `/data/postgres/data`, создаёт собственную systemd-службу, пользователей, базу и postgres-exporter.

In [4]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible-playbook postgres.play.yml


PLAY [Install PostgreSQL on vm1] ***********************************************

TASK [Gathering F

acts] *********************************************************


ok: [vm1]

TASK [postgres : Install PostgreSQL packages] **********************************


ok: [vm1]

TASK [postgres : Stop and disable the package-created cluster] *****************


ok: [vm1]

TASK [postgres : Create /data directory] ***************************************


...

ok: [vm1]

RUNNING HANDLER [postgres : Restart postgres-exporter] *************************


changed: [vm1]

TASK [postgres : Enable and start postgres-exporter service] *******************


changed: [vm1]

PLAY RECAP *********************************************************************
vm1

                        : ok=28   changed=20   unreachable=0    failed=0    skipped=0    rescued=0  

  ignored=0   



## 4. Повторный запуск — проверка идемпотентности

При повторном запуске ожидается `changed=0` в `PLAY RECAP`.

In [5]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible-playbook postgres.play.yml


PLAY [Install PostgreSQL on vm1] ***********************************************

TASK [Gathering F

acts] *********************************************************


ok: [vm1]

TASK [postgres : Install PostgreSQL packages] **********************************


ok: [vm1]

TASK [postgres : Stop and disable the package-created cluster] *****************


ok: [vm1]

TASK [postgres : Create /data directory] ***************************************


...

ok: [vm1]

TASK [postgres : Apply exporter handlers before health checks] *****************

TASK [p

ostgres : Enable and start postgres-exporter service] *******************


ok: [vm1]

PLAY RECAP *********************************************************************
vm1     

                   : ok=23   changed=0    unreachable=0    failed=0    skipped=1    rescued=0    ign

ored=0   



## 5. Проверка результата

In [6]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible vm1 -b -m command -a 'systemctl is-active postgresql-hw2 postgres-exporter'
ansible vm1 -b -m command -a 'systemctl is-enabled postgresql-hw2 postgres-exporter'

vm1 | CHANGED | rc=0 >>
active
active


vm1 | CHANGED | rc=0 >>
enabled
enabled


In [7]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible vm1 -b --become-user postgres -m command -a 'psql -c "SELECT 1"'
ansible vm1 -b --become-user postgres -m command -a 'psql -c "\l"'
ansible vm1 -b --become-user postgres -m command -a 'psql -c "\du"'

vm1 | CHANGED | rc=0 >>
 ?column? 
----------
        1
(1 row)


vm1 | CHANGED | rc=0 >>
                                                   List of databases
   Name

    |  Owner   | Encoding | Locale Provider | Collate |  Ctype  | ICU Locale | ICU Rules |   Access 

privileges   
-----------+----------+----------+-----------------+---------+---------+------------+-

----------+-----------------------
 app_db    | app_user | UTF8     | libc            | C.UTF-8 | C.

...

s=CTc/postgres
(4 rows)


vm1 | CHANGED | rc=0 >>
                                 List of roles
     Role name     |         

                Attributes                         
-------------------+----------------------------

--------------------------------
 app_user          | 
 postgres          | Superuser, Create role, 

Create DB, Replication, Bypass RLS
 postgres_exporter | 


In [8]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
ansible vm1 -m shell -a "curl -s http://127.0.0.1:9187/metrics | grep '^pg_up '"
ansible vm1 -b -m shell -a "stat -c '%U:%G %a %n' /data /data/postgres /data/postgres/data"

vm1 | CHANGED | rc=0 >>
pg_up 1


vm1 | CHANGED | rc=0 >>
root:root 755 /data
postgres:postgres 700 /data/postgres
postgres:postgres 7

00 /data/postgres/data


In [9]:
%%bash
set -e
cd /mnt/hgfs/hw/HW2
head -n 1 inventories/main/group_vars/all/vault.yml
ansible-vault view inventories/main/group_vars/all/vault.yml | sed -E 's/:.*/: <encrypted>/'

$ANSIBLE_VAULT;1.1;AES256


---
vault_postgres_app_password: <encrypted>
vault_postgres_exporter_password: <encrypted>



## Вывод

Собственная роль развернула PostgreSQL в параметризованном каталоге, применила конфигурацию из словаря и правила авторизации из списка, создала пользовательские объекты и запустила postgres-exporter. Повторный запуск с `changed=0` подтверждает идемпотентность. Пароли пользователей БД не хранятся в открытом виде.